# Tiny RDT-MLP on MNIST

Compare a simple MLP baseline with a recurrent-depth MLP that reuses the same gated block. The main question: does increasing `steps` improve MNIST validation accuracy enough to justify recurrent depth?

Pipeline: `28x28 grayscale -> flatten 784 -> hidden embedding -> repeated gated MLP block -> 10-class classifier`.

In [ ]:
# Imports and reproducibility
from __future__ import annotations

import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Using device: {device}')

## Load MNIST

This cell downloads MNIST if it is not already available under `./data`. It reserves 10,000 training samples for validation.

In [ ]:
BATCH_SIZE = 128
VALIDATION_SIZE = 10_000

transform = transforms.ToTensor()
full_train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [len(full_train_dataset) - VALIDATION_SIZE, VALIDATION_SIZE],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f'train={len(train_dataset):,}, val={len(val_dataset):,}, test={len(test_dataset):,}')

## Models

`TinyRDTMLPMNIST` shares one `GatedMLPBlock` across every recurrent step. Increasing `steps` changes computation depth without adding another block's parameters.

In [ ]:
class SimpleMLPMNIST(nn.Module):
    def __init__(self, input_dim: int = 28 * 28, hidden_dim: int = 128, num_classes: int = 10):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.network(x)


class GatedMLPBlock(nn.Module):
    def __init__(self, hidden_dim: int, mlp_dim: int, gate_bias: float = -2.0):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_dim)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, hidden_dim),
        )
        self.gate = nn.Linear(hidden_dim * 2, hidden_dim)
        nn.init.constant_(self.gate.bias, gate_bias)

    def forward(self, h):
        candidate = self.mlp(self.norm(h))
        gate = torch.sigmoid(self.gate(torch.cat([h, candidate], dim=-1)))
        return gate * candidate + (1.0 - gate) * h


class TinyRDTMLPMNIST(nn.Module):
    def __init__(self, input_dim: int = 28 * 28, hidden_dim: int = 128, mlp_dim: int = 256, num_classes: int = 10, steps: int = 8):
        super().__init__()
        self.steps = steps
        self.input_proj = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.prelude = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.LayerNorm(hidden_dim))
        self.block = GatedMLPBlock(hidden_dim=hidden_dim, mlp_dim=mlp_dim)
        self.coda = nn.Sequential(nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, hidden_dim), nn.GELU())
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        h = self.prelude(self.input_proj(x))
        for _ in range(self.steps):
            h = self.block(h)
        return self.classifier(self.coda(h))

## Training helpers

Each experiment records `loss`, `val_loss`, `accuracy`, and `val_accuracy` once per epoch.

In [ ]:
@dataclass
class Metrics:
    loss: float
    accuracy: float


def run_epoch(model, loader, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.set_grad_enabled(is_training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = F.cross_entropy(logits, labels)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += labels.size(0)

    return Metrics(loss=total_loss / total_samples, accuracy=total_correct / total_samples)


def train_model(model, epochs=5, lr=1e-3, weight_decay=1e-4):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}

    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(model, train_loader, optimizer)
        val_metrics = run_epoch(model, val_loader)
        history['loss'].append(train_metrics.loss)
        history['val_loss'].append(val_metrics.loss)
        history['accuracy'].append(train_metrics.accuracy)
        history['val_accuracy'].append(val_metrics.accuracy)
        print(
            f'epoch={epoch:02d} loss={train_metrics.loss:.4f} val_loss={val_metrics.loss:.4f} '
            f'acc={train_metrics.accuracy:.4f} val_acc={val_metrics.accuracy:.4f}'
        )

    return model, history

## Run comparison

The default sweep follows the suggested comparison: simple MLP and RDT with `steps=1/4/8/16`. Set `EPOCHS = 5` to `10` for the intended experiment. For a quick smoke test, temporarily use `EPOCHS = 1`.

In [ ]:
EPOCHS = 5
HIDDEN_DIM = 128
MLP_DIM = 256

experiments = {
    'Simple MLP': lambda: SimpleMLPMNIST(hidden_dim=HIDDEN_DIM),
    'RDT steps=1': lambda: TinyRDTMLPMNIST(hidden_dim=HIDDEN_DIM, mlp_dim=MLP_DIM, steps=1),
    'RDT steps=4': lambda: TinyRDTMLPMNIST(hidden_dim=HIDDEN_DIM, mlp_dim=MLP_DIM, steps=4),
    'RDT steps=8': lambda: TinyRDTMLPMNIST(hidden_dim=HIDDEN_DIM, mlp_dim=MLP_DIM, steps=8),
    'RDT steps=16': lambda: TinyRDTMLPMNIST(hidden_dim=HIDDEN_DIM, mlp_dim=MLP_DIM, steps=16),
}

trained_models = {}
histories = {}
for name, build_model in experiments.items():
    print(f'\n=== {name} ===')
    trained_models[name], histories[name] = train_model(build_model(), epochs=EPOCHS)

## Plot training and validation metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
epochs = range(1, EPOCHS + 1)

for name, history in histories.items():
    axes[0].plot(epochs, history['accuracy'], marker='o', label=f'{name} train')
    axes[0].plot(epochs, history['val_accuracy'], marker='o', linestyle='--', label=f'{name} val')
    axes[1].plot(epochs, history['loss'], marker='o', label=f'{name} train')
    axes[1].plot(epochs, history['val_loss'], marker='o', linestyle='--', label=f'{name} val')

axes[0].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[1].set(title='Loss', xlabel='Epoch', ylabel='Cross-entropy loss')
for axis in axes:
    axis.grid(alpha=0.3)
    axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Final comparison

Use test accuracy only after choosing a configuration based on validation metrics.

In [ ]:
summary = []
for name, model in trained_models.items():
    test_metrics = run_epoch(model, test_loader)
    summary.append({
        'model': name,
        'parameters': sum(parameter.numel() for parameter in model.parameters()),
        'final_val_accuracy': histories[name]['val_accuracy'][-1],
        'test_accuracy': test_metrics.accuracy,
        'test_loss': test_metrics.loss,
    })

for result in sorted(summary, key=lambda item: item['test_accuracy'], reverse=True):
    print(result)

## Interpretation checklist

- Compare `RDT steps=1` against `steps=4/8/16`.
- If extra steps do not improve validation accuracy, MNIST likely does not need recurrent depth.
- Compare parameter counts: RDT step count increases repeated computation, not block parameter count.
- Check validation loss as well as accuracy to identify overfitting.